# Notebook 2 — Analisis Linguistic Marker dan Uji Signifikansi
## Gender Persona × GSM8K

**Peneliti:** Abel Apriliani (18222008) — Tugas Akhir S1 Informatika ITB
**Model contoh:** Meta Llama 3.1 8B Instruct

---

### Tujuan Notebook

Notebook ini mengukur **penanda linguistik (linguistic markers)** pada respons model dan menguji apakah kondisi persona gender menghasilkan perbedaan penanda yang signifikan secara statistik.

Berbeda dari akurasi (yang biner: benar/salah), penanda linguistik adalah **kepadatan (density)** — jumlah kemunculan penanda per 100 kata. Ini variabel kontinu, sehingga uji statistiknya berbeda dari Notebook 1.

### Kerangka Penanda (dengan anchor paper per grup)

Setiap grup penanda diambil dari literatur mapan agar dapat dipertanggungjawabkan di sidang:

| Grup | Penanda | Anchor Paper |
|---|---|---|
| A. Opening Ritual | cara membuka jawaban | Swales (1990) — CARS moves |
| B. Reasoning Organization | langkah bernomor, bullet | Wei et al. (2022) — Chain-of-Thought |
| C. Epistemic Stance | hedging, booster | Hyland (2005) — Metadiscourse |
| D. Inferential Connective | konektif logis vs informal | Halliday & Hasan (1976); Park & Lei (2026) |
| E. Mathematical Formatting | LaTeX, aritmetika polos | (operasional, domain GSM8K) |
| F. Interactional | inclusive "we", first-person | Hyland (2005) — Metadiscourse |
| — Verbosity | jumlah kata | Meek et al. (2025) |


## Bagian 1 — Setup dan Ekstraksi Penanda

In [ ]:
# ── 1.1 Imports ──────────────────────────────────────────────────────────────
import re, warnings
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal, mannwhitneyu, shapiro
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=0.95)
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
print("Library siap.")


In [ ]:
# ── 1.2 Konfigurasi & pemuatan ───────────────────────────────────────────────
MODEL_FILES = {
    "Llama-3.1-8B": "results-meta-llama-3_1-8b-instruct.csv",
    # tambahkan model lain di sini
}
ACTIVE_MODEL = "Llama-3.1-8B"
TEXT_COL = "response_content_full"   # kolom teks respons GSM8K
PERSONA_ORDER = [
    "man_explicit_1","man_implicit_1","man_implicit_2","man_implicit_3",
    "woman_explicit_1","woman_implicit_1","woman_implicit_2","woman_implicit_3",
]
OUTPUT_DIR = Path("outputs_notebook2"); OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(MODEL_FILES[ACTIVE_MODEL], low_memory=False)
df.columns = df.columns.str.strip()
df["qid"] = df["file"].str.extract(r"gsm8k_(\d+)").astype(int)
df["text"] = df[TEXT_COL].fillna("").astype(str)
df["gender"] = df["persona_folder"].apply(lambda p: "man" if str(p).startswith("man") else ("woman" if str(p).startswith("woman") else "neutral"))
df["persona_type"] = df["persona_folder"].apply(lambda p: "explicit" if "explicit" in str(p) else ("implicit" if "implicit" in str(p) else "neutral"))
personas = [p for p in PERSONA_ORDER if p in df["persona_folder"].unique()]
if "neutral" in df["persona_folder"].unique(): personas += ["neutral"]
print(f"Data: {len(df):,} baris | {len(personas)} persona")
print(f"Kolom teks: {TEXT_COL}")


In [ ]:
# ── 1.3 Definisi penanda linguistik (regex, per grup + anchor) ───────────────
# Setiap penanda dihitung sebagai KEPADATAN: jumlah match / jumlah kata * 100.
MARKERS = {
    # Grup B — Reasoning Organization (Wei et al. 2022)
    "B_explicit_step": (r"\*{1,2}Step\s*\d+|\bStep\s+\d+[:.]", "Langkah bernomor eksplisit"),
    "B_bullet":        (r"(?m)^\s*[-*\u2022]\s+\w", "Poin bullet"),
    # Grup C — Epistemic Stance (Hyland 2005)
    "C_hedging":       (r"\b(maybe|perhaps|might|possibly|it seems|i think)\b", "Hedging (ketidakpastian)"),
    "C_booster":       (r"\b(clearly|obviously|definitely|certainly|of course)\b", "Booster (penegasan)"),
    # Grup D — Inferential Connective (Halliday & Hasan 1976; Park & Lei 2026)
    "D_formal_conn":   (r"\b(therefore|thus|hence|consequently)\b", "Konektif logis formal"),
    "D_informal_conn": (r"(?<!\w)so\s+(?:she|he|they|it|we|the|each)\b", "Konektif informal (so ...)"),
    # Grup E — Mathematical Formatting (operasional domain GSM8K)
    "E_latex":         (r"\\\[[\s\S]{1,400}?\\\]|\$\$[\s\S]{1,400}?\$\$|\\\([^)]{1,150}\\\)", "Notasi LaTeX"),
    "E_plain_arith":   (r"\b\d[\d,\.]*\s*[+\-\u00d7x\u00f7/]\s*\d[\d,\.]*\s*=\s*\d", "Aritmetika polos"),
    "E_boxed":         (r"\\boxed\{", "Boxed answer"),
    # Grup F — Interactional (Hyland 2005)
    "F_inclusive_we":  (r"\b(we\s+(?:need|can|get|have|find|calculate)|let'?s|let us)\b", "Inclusive we"),
    "F_first_i":       (r"\b(i\s+(?:think|need|get|find|calculate|will))\b", "First-person I"),
}

def word_count(t): return max(len(re.findall(r"\b\w+\b", t)), 1)
df["wc"] = df["text"].apply(word_count)

for name, (pat, _) in MARKERS.items():
    rgx = re.compile(pat, re.IGNORECASE)
    df[name + "_d"] = df["text"].apply(lambda t: len(rgx.findall(t))) / df["wc"] * 100

# Verbosity sebagai penanda tersendiri (Meek et al. 2025)
DENSITY_COLS = [name + "_d" for name in MARKERS] + ["wc"]
print("Penanda dihitung:")
for name,(_,desc) in MARKERS.items(): print(f"  {name}_d : {desc}")
print("  wc     : Jumlah kata (verbosity, Meek et al. 2025)")


In [ ]:
# ── 1.4 Profil penanda per persona (mean density) ────────────────────────────
profile = df.groupby("persona_folder")[DENSITY_COLS].mean().reindex(personas).round(4)
print("=== Rata-rata Kepadatan Penanda per Persona ===")
print(profile.to_string())
profile.to_csv(OUTPUT_DIR / "profil_penanda.csv")


## Bagian 2 — Justifikasi Non-Parametrik (Shapiro-Wilk)

Sebelum memilih uji, normalitas kepadatan penanda diperiksa. Kepadatan penanda umumnya **tidak normal**: banyak respons bernilai nol (penanda tidak muncul) dan sedikit respons bernilai tinggi. Distribusi seperti ini melanggar asumsi ANOVA/t-test.

**Shapiro-Wilk** (Shapiro & Wilk, 1965) menguji normalitas. Bila H0 (normal) ditolak, uji non-parametrik (Kruskal-Wallis, Mann-Whitney U) menjadi pilihan yang tepat.


In [ ]:
# ── 2.1 Uji Shapiro-Wilk pada setiap penanda ─────────────────────────────────
print("=== Shapiro-Wilk per Penanda (sampel maks 5000) ===")
print("H0: normal. p<0.05 -> tidak normal -> non-parametrik.\n")
norm_rows = []
for col in DENSITY_COLS:
    vals = df[col].dropna()
    sample = vals.sample(min(5000, len(vals)), random_state=42)
    if sample.nunique() < 3:
        norm_rows.append({"penanda": col, "W": np.nan, "p": np.nan, "kesimpulan": "konstan/jarang"})
        continue
    W, p = shapiro(sample)
    norm_rows.append({"penanda": col, "W": round(W,3), "p": f"{p:.2e}",
                      "kesimpulan": "tidak normal" if p<0.05 else "normal"})
norm_df = pd.DataFrame(norm_rows)
print(norm_df.to_string(index=False))
print("\nMayoritas penanda tidak normal -> Kruskal-Wallis & Mann-Whitney U digunakan.")


## Bagian 3 — Kruskal-Wallis (Uji Omnibus per Penanda)

**Alasan pemilihan:** Kruskal-Wallis (Kruskal & Wallis, 1952) adalah alternatif non-parametrik dari ANOVA satu arah. Ia menguji apakah **setidaknya satu** persona berbeda dari yang lain pada suatu penanda, tanpa mengasumsikan normalitas. Digunakan sebagai uji omnibus sebelum perbandingan pasangan (agar tidak langsung menguji semua pasangan).


In [ ]:
# ── 3.1 Kruskal-Wallis per penanda ───────────────────────────────────────────
print("=== Kruskal-Wallis per Penanda (antar semua persona) ===\n")
kw_rows, kw_pvals = [], []
for col in DENSITY_COLS:
    groups = [df[df["persona_folder"]==p][col].dropna().values for p in personas]
    groups = [g for g in groups if len(g) > 0]
    try:
        H, p = kruskal(*groups)
    except Exception:
        H, p = np.nan, np.nan
    kw_rows.append({"penanda": col, "H": round(H,3) if not np.isnan(H) else np.nan, "p_raw": p})
    kw_pvals.append(p if not np.isnan(p) else 1.0)

# Koreksi Bonferroni untuk banyaknya penanda diuji
rej, p_bonf, _, _ = multipletests(kw_pvals, method="bonferroni")
kw_df = pd.DataFrame(kw_rows)
kw_df["p_bonferroni"] = np.round(p_bonf, 5)
kw_df["signifikan"] = rej
kw_df = kw_df.sort_values("p_raw")
print(kw_df.to_string(index=False))
kw_df.to_csv(OUTPUT_DIR / "kruskal_wallis.csv", index=False)
print("\nPenanda dengan signifikan=True berbeda nyata antar persona setelah koreksi Bonferroni.")


## Bagian 4 — Mann-Whitney U (Perbandingan Pasangan Terarah)

**Alasan pemilihan:** Untuk penanda yang signifikan pada Kruskal-Wallis, Mann-Whitney U (Mann & Whitney, 1947) menunjukkan **pasangan persona mana** yang berbeda. Digunakan pada pasangan berhipotesis (explicit vs implicit, man vs woman), bukan semua kombinasi.

**Effect size:** dua ukuran dilaporkan agar kuat —
- **Rank-biserial r** (Kerby, 2014): turunan langsung dari statistik U.
- **Cliff's delta** (Cliff, 1993): ukuran dominasi non-parametrik, kokoh untuk distribusi tidak normal.

**Koreksi:** Bonferroni untuk mengendalikan family-wise error.


In [ ]:
# ── 4.1 Fungsi effect size ───────────────────────────────────────────────────
def rank_biserial(U, n1, n2):
    return 1 - (2 * U) / (n1 * n2)

def cliffs_delta(a, b):
    """Cliff's delta via perbandingan pasangan (vektorisasi)."""
    a = np.asarray(a); b = np.asarray(b)
    # untuk efisiensi, sampel bila terlalu besar
    if len(a) * len(b) > 4_000_000:
        rng = np.random.default_rng(42)
        a = rng.choice(a, size=2000, replace=False) if len(a) > 2000 else a
        b = rng.choice(b, size=2000, replace=False) if len(b) > 2000 else b
    gt = sum((a[:, None] > b[None, :]).sum(axis=1))
    lt = sum((a[:, None] < b[None, :]).sum(axis=1))
    return (gt - lt) / (len(a) * len(b))

def interpret_cliff(d):
    a = abs(d)
    return ("besar" if a>=0.474 else "sedang" if a>=0.33 else "kecil" if a>=0.147 else "dapat diabaikan")

HIPOTESIS_PAIRS = [
    ("man_explicit_1","man_implicit_1"),
    ("woman_explicit_1","woman_implicit_1"),
    ("man_explicit_1","woman_explicit_1"),
    ("man_implicit_1","woman_implicit_1"),
]
if "neutral" in personas:
    HIPOTESIS_PAIRS += [("neutral","man_explicit_1"),("neutral","woman_explicit_1")]
print("Pasangan yang diuji:")
for a,b in HIPOTESIS_PAIRS: print(f"  {a} vs {b}")


In [ ]:
# ── 4.2 Mann-Whitney U untuk penanda signifikan ──────────────────────────────
# Fokus ke penanda yang signifikan di Kruskal-Wallis (agar tidak menguji yang tak berbeda)
signif_markers = kw_df[kw_df["signifikan"]]["penanda"].tolist()
if not signif_markers:
    signif_markers = kw_df.nsmallest(5, "p_raw")["penanda"].tolist()
    print("Tidak ada yang lolos Bonferroni; menggunakan 5 penanda dengan p terkecil.\n")
print(f"Penanda diuji pairwise: {signif_markers}\n")

mwu_rows, mwu_pvals = [], []
for col in signif_markers:
    for a, b in HIPOTESIS_PAIRS:
        va = df[df["persona_folder"]==a][col].dropna().values
        vb = df[df["persona_folder"]==b][col].dropna().values
        if len(va)==0 or len(vb)==0: continue
        U, p = mannwhitneyu(va, vb, alternative="two-sided")
        r = rank_biserial(U, len(va), len(vb))
        d = cliffs_delta(va, vb)
        mwu_rows.append({"penanda": col, "A": a, "B": b,
                         "mean_A": round(va.mean(),4), "mean_B": round(vb.mean(),4),
                         "U": U, "p_raw": p, "rank_biserial": round(r,3),
                         "cliffs_delta": round(d,3), "effect": interpret_cliff(d)})
        mwu_pvals.append(p)

rej, p_bonf, _, _ = multipletests(mwu_pvals, method="bonferroni")
mwu_df = pd.DataFrame(mwu_rows)
mwu_df["p_bonferroni"] = np.round(p_bonf, 5)
mwu_df["signifikan"] = rej
print("=== Mann-Whitney U Pairwise (koreksi Bonferroni) ===")
print(mwu_df.to_string(index=False))
mwu_df.to_csv(OUTPUT_DIR / "mannwhitney_pairwise.csv", index=False)


In [ ]:
# ── 4.3 Visualisasi penanda kunci per persona ────────────────────────────────
top_markers = kw_df.nsmallest(6, "p_raw")["penanda"].tolist()
fig, axes = plt.subplots(2, 3, figsize=(16, 9)); axes = axes.flatten()
GENDER_COLORS = {"man":"#2E86C1","woman":"#CB4335","neutral":"#7F8C8D"}
for ax, col in zip(axes, top_markers):
    means = df.groupby("persona_folder")[col].mean().reindex(personas)
    colors = [GENDER_COLORS["man" if p.startswith("man") else "woman" if p.startswith("woman") else "neutral"] for p in personas]
    ax.bar(range(len(means)), means.values, color=colors, edgecolor="white")
    ax.set_xticks(range(len(means)))
    ax.set_xticklabels([p.replace("_","\n") for p in personas], fontsize=7)
    ax.set_title(col, fontweight="bold", fontsize=10)
    ax.set_ylabel("density /100 kata")
plt.suptitle(f"Penanda Linguistik Kunci per Persona — {ACTIVE_MODEL}", fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_penanda_kunci.png", dpi=150, bbox_inches="tight")
plt.show()


## Bagian 5 — Ringkasan Naratif untuk Bab VI

In [ ]:
# ── 5.1 Naratif sumber (siap salin ke Bab VI) ────────────────────────────────
narasi = """
RINGKASAN METODE UJI PENANDA LINGUISTIK — untuk Bab VI Tugas Akhir
===================================================================

VARIABEL: kepadatan penanda linguistik (jumlah kemunculan per 100 kata).
Berbeda dari akurasi yang biner, kepadatan bersifat kontinu namun tidak
berdistribusi normal (banyak nilai nol, ekor kanan panjang).

1. PEMERIKSAAN NORMALITAS — Shapiro-Wilk (Shapiro & Wilk, 1965)
   Mayoritas penanda menyimpang dari normal, sehingga uji non-parametrik dipilih.

2. UJI OMNIBUS — Kruskal-Wallis (Kruskal & Wallis, 1952)
   Alternatif non-parametrik ANOVA. Menguji apakah minimal satu persona berbeda
   pada tiap penanda. Koreksi Bonferroni diterapkan karena banyak penanda diuji.

3. PERBANDINGAN PASANGAN — Mann-Whitney U (Mann & Whitney, 1947)
   Untuk penanda yang signifikan, mengidentifikasi pasangan persona yang berbeda.
   Dua effect size dilaporkan: rank-biserial correlation (Kerby, 2014) dan
   Cliff's delta (Cliff, 1993). Koreksi Bonferroni untuk perbandingan ganda.

CATATAN PENTING (untuk pertahanan sidang):
   Dengan ukuran sampel besar, hampir semua uji cenderung signifikan. Karena itu
   EFFECT SIZE (bukan hanya p-value) menjadi dasar interpretasi. Penanda dinyatakan
   berbeda secara bermakna hanya bila signifikan DAN memiliki effect size minimal
   'kecil' menurut ambang Cliff's delta.

KERANGKA PENANDA — anchor paper:
   Swales (1990) — Opening ritual (CARS moves)
   Wei et al. (2022) — Chain-of-Thought (reasoning organization)
   Hyland (2005) — Metadiscourse (epistemic stance, interactional)
   Halliday & Hasan (1976); Park & Lei (2026) — Inferential connectives
   Meek et al. (2025) — Verbosity

DAFTAR SUMBER UJI STATISTIK:
   Cliff, N. (1993). Dominance statistics: Ordinal analyses.
   Kerby, D. S. (2014). The simple difference formula.
   Kruskal, W. H., & Wallis, W. A. (1952). Use of ranks in one-criterion variance analysis.
   Mann, H. B., & Whitney, D. R. (1947). On a test of whether one of two random
     variables is stochastically larger than the other.
   Shapiro, S. S., & Wilk, M. B. (1965). An analysis of variance test for normality.
"""
print(narasi)
with open(OUTPUT_DIR / "narasi_bab6_penanda.txt","w") as f: f.write(narasi)
print("Disimpan: narasi_bab6_penanda.txt")


In [ ]:
# ── 5.2 Daftar output ────────────────────────────────────────────────────────
print("Output tersimpan di:", OUTPUT_DIR)
for f in sorted(OUTPUT_DIR.glob("*")): print(f"  - {f.name}")
print("\nNotebook 2 selesai: ekstraksi penanda -> Shapiro-Wilk -> Kruskal-Wallis")
print("-> Mann-Whitney U pairwise -> effect size (rank-biserial + Cliff's delta).")
